# 00 — Prepare the canonical object for the paper figures

Builds **one** object, `Data/ComboScreen_processed.h5ad`, straight from the raw
`ComboScreen.h5ad`, following the same steps as
`Src/ComboScreen/01_Data_outlook.ipynb` — but **keeping the raw counts as a
layer** so every downstream analysis can pull whatever representation it needs.

Layer / slot convention in the saved object:
- `.layers['counts']` — raw UMI counts            (QC, re-normalisation)
- `.X` and `.raw`     — log1p-normalised expression (gene plots, scoring)
- `.obsm`             — `X_pca`, `X_umap`, `X_diffmap` (scaling is transient, only
  used to compute PCA/UMAP, and is **not** kept as `.X`)
- `.obs`              — labels + `state` (NEPC-N/NEPC-A&N/NEPC-A) + `perturbation*`
  + `Doxo1program_score` + `dpt_pseudotime`

Run this **once** (memory-heavy). Afterwards every figure notebook just does
`fu.load("processed")`.

In [ ]:
import _figutils as fu
import importlib; importlib.reload(fu)
import scanpy as sc
import numpy as np
import pandas as pd

fu.set_theme()
sc.settings.verbosity = 1

TARGET_SUM = 20_000      # matches 01_Data_outlook
N_HVG = 3000
SCALE_MAX = 9
N_PCS = 50
N_NEIGHBORS = 8
NEIGHBOR_METRIC = "euclidean"


### Load raw counts and select cells (same filter as 01_Data_outlook)

In [ ]:
adata = fu.load("raw")          # ComboScreen.h5ad: 449k cells, raw counts in .X
print("loaded:", adata.shape)

# Keep cells with fewer than 3 of the guide one-hots set (drops 3+ guide cells).
keep = adata.obs[fu.GUIDE_COLS].sum(axis=1) < 3
adata = adata[keep].copy()
print("after sum<3 filter:", adata.shape)


### Labels: perturbation, state, time

In [ ]:
adata.obs["perturbation"] = fu.combine_onehot(adata)
adata.obs["perturbation_clean"] = fu.clean_perturbation_labels(adata.obs["perturbation"])
adata.obs["perturbation_time"] = (
    adata.obs["perturbation"].astype(str) + "_" + adata.obs["time_point"].astype(str))

# 3-state manuscript label from final_label, ordered along the trajectory.
fu.apply_state_naming(adata)
fu.set_state_categories(adata)
adata.obs["time_point"] = pd.Categorical(
    adata.obs["time_point"].astype(str), categories=fu.TIMEPOINT_ORDER, ordered=True)
if "conditions" in adata.obs:
    adata.obs["conditions"] = adata.obs["conditions"].astype(str).str.strip()

print(pd.crosstab(adata.obs["final_label"], adata.obs["state"]))


### Keep raw counts as a layer, then normalise + log1p

In [ ]:
# Raw counts -> layer (X is still raw counts at this point).
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
sc.pp.log1p(adata)
adata.raw = adata               # .raw = log-normalised (use_raw=True works in sc.pl)
# .X is now log-normalised and will stay that way in the saved object.


### Doxo1 differentiation score (on log-normalised data)

In [ ]:
doxo1 = pd.read_csv(fu.DOXO1_SIGNATURE_CSV, index_col=0)
doxo1_genes = [g for g in doxo1["names"].astype(str) if g in adata.var_names]
print(f"Doxo1 signature: {len(doxo1_genes)}/{len(doxo1)} genes found")
sc.tl.score_genes(adata, gene_list=doxo1_genes,
                  score_name="Doxo1program_score", use_raw=False)


### Embedding — scale transiently for PCA/UMAP

Scaling is needed only to compute PCA/UMAP; we do it on a copy so the saved `.X`
stays log-normalised. The embeddings (`X_pca`, `X_umap`) and graph are copied back.

In [ ]:
scaled = adata.copy()
sc.pp.highly_variable_genes(scaled, n_top_genes=N_HVG)
sc.pp.scale(scaled, max_value=SCALE_MAX)
sc.pp.pca(scaled, n_comps=N_PCS, svd_solver="arpack")
sc.pp.neighbors(scaled, n_neighbors=N_NEIGHBORS, metric=NEIGHBOR_METRIC, n_pcs=N_PCS)
sc.tl.umap(scaled)

# Copy embedding + graph back onto the log-normalised object.
adata.obsm["X_pca"] = scaled.obsm["X_pca"]
adata.obsm["X_umap"] = scaled.obsm["X_umap"]
adata.varm["PCs"] = scaled.varm["PCs"]
adata.obsp["distances"] = scaled.obsp["distances"]
adata.obsp["connectivities"] = scaled.obsp["connectivities"]
adata.uns["neighbors"] = scaled.uns["neighbors"]
adata.uns["pca"] = scaled.uns["pca"]
adata.uns["umap"] = scaled.uns.get("umap", {})
adata.var["highly_variable"] = (
    scaled.var["highly_variable"].reindex(adata.var_names).fillna(False))
print("embedding computed:", adata.obsm["X_umap"].shape)


### Diffusion pseudotime (root = least-differentiated cell)

Root chosen from the Doxo1 score: the diffusion-space medoid of the bottom 1%
(least-differentiated) cells. Pseudotime oriented to increase NEPC-N -> NEPC-A.

In [ ]:
sc.tl.diffmap(scaled, n_comps=15)
adata.obsm["X_diffmap"] = scaled.obsm["X_diffmap"]
adata.uns["diffmap_evals"] = scaled.uns["diffmap_evals"]

score = adata.obs["Doxo1program_score"].values
thr = np.quantile(score, 0.01)
cand = np.where(score <= thr)[0]
dm = adata.obsm["X_diffmap"]
centroid = dm[cand].mean(axis=0)
root = int(cand[np.argmin(np.linalg.norm(dm[cand] - centroid, axis=1))])
adata.uns["iroot"] = root
scaled.uns["iroot"] = root
print(f"root cell {root} | state {adata.obs[fu.CELL_STATE_COL].iloc[root]} | "
      f"Doxo1 {score[root]:.3f} (thr {thr:.3f})")

sc.tl.dpt(scaled, n_dcs=10)
pt = scaled.obs["dpt_pseudotime"].values.copy()
inf = np.isinf(pt)
if inf.any():
    pt[inf] = pt[~inf].max()
adata.obs["dpt_pseudotime"] = pt

# Orient NEPC-N (low) -> NEPC-A (high).
present = fu.ordered_states(adata)
med = (adata.obs.groupby(fu.CELL_STATE_COL, observed=True)["dpt_pseudotime"]
       .median().reindex(present))
if med.iloc[0] > med.iloc[-1]:
    print("flipping pseudotime direction")
    adata.obs["dpt_pseudotime"] = adata.obs["dpt_pseudotime"].max() - adata.obs["dpt_pseudotime"]
del scaled
(adata.obs.groupby(fu.CELL_STATE_COL, observed=True)["dpt_pseudotime"]
 .median().reindex(present))


### Palettes and save

In [ ]:
adata.uns["state_colors"] = [fu.STATE_PALETTE[c] for c in adata.obs[fu.CELL_STATE_COL].cat.categories]
adata.uns["time_point_colors"] = [fu.TIME_PALETTE[c] for c in adata.obs["time_point"].cat.categories]

adata.write(fu.PROCESSED_H5AD)
print("wrote", fu.PROCESSED_H5AD)
print(adata)
print("layers:", list(adata.layers.keys()))
